In [8]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.abstract_event_listener import AbstractEventListener
from selenium.webdriver.support.events import EventFiringWebDriver, AbstractEventListener
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import UnexpectedAlertPresentException
from selenium import webdriver
from webdriver_auto_update.chrome_app_utils import ChromeAppUtils
from webdriver_auto_update.webdriver_manager import WebDriverManager
import traceback

#* required
import base64
import re
import os

#* setup
def setup_chrome():
    options = Options()
    options.add_experimental_option("debuggerAddress", "localhost:8989")
    driver = webdriver.Chrome(service=Service(r'C:\bin\chromedriver.exe'), options=options)
    return driver

def get_tabs():
    global merged_dict
    try:
        # if parent.winfo_exists():
        if True:
            print("รายงานจำนวนtabs")

            # * เก็บชื่อ title และ value ของ tab ที่เปิดอยู่
            title_list = []
            # title_list_Idx = [] #!เหมือนจะไม่ได้ใช้
            value_list = []
            # title_dict = {} #!เหมือนจะไม่ได้ใช้
            for idx, handle in enumerate(driver.window_handles):
                driver.switch_to.window(handle)
                # title_list_Idx.append(
                #     driver.title + "["+str(idx)+"]") #!เหมือนจะไม่ได้ใช้
                title_list.append(driver.title)

                value_list.append(driver.current_window_handle)
                # title_dict.update(
                #     {driver.title: driver.current_window_handle}) #!เหมือนจะไม่ได้ใช้

            # * เอาtitle มาทำให้ unique เพราะ title จะสามารถที่จะซ้ำกันได้
            unique_titles = []
            counter = {}
            for item in title_list:
                if item in counter:
                    counter[item] += 1
                    print("counter[item] คือไร: ", counter[item])
                    unique_titles.append(
                        f"{item}{counter[item]-1}")
                else:
                    counter[item] = 1
                    unique_titles.append(item)

            # * เอาList มารวมกัน
            merged_dict = dict(zip(unique_titles, value_list))
            print("มี tabs ไรบ้าง", merged_dict)
            
    except Exception as e:
        traceback_str = traceback.format_exc()
        print(f"An error occirred: {e}")
        print(traceback_str)
        
def fill_items(array_items=[]):
    sku_input_xpath = '/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[1]/from/div/div/div[1]/div[1]/span/input'
    
    for item in array_items:
        driver.find_element(By.XPATH, sku_input_xpath).clear()
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(item)
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(Keys.ENTER)

def embed_size_check():
    embed_element = driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div[2]/div/div[2]/div[2]/div/embed")
    pdf_src = driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div[2]/div/div[2]/div[2]/div/embed").get_attribute('src')
    proc = re.search("(?<=,).*", pdf_src)
    base64_pdf_data = proc.group(0)
    bin_pdf_data = base64.b64decode(base64_pdf_data) #แปลง base64 to binary data
    with open("output.pdf", "wb") as pdf_file:
        pdf_file.write(bin_pdf_data)
    os.startfile("output.pdf", "print")
    embed_size = embed_element.size
    print(embed_size)
    print(pdf_src)
    print(f"base64 pdf extracted: {base64_pdf_data}")

driver = setup_chrome()
get_tabs()
driver.switch_to.window(merged_dict['SMCO :: พิมพ์ใบเสร็จซ้ำ'])
#* เอา function ที่ต้องการเทสมาใส่ข้างล่างนี่

#* function1
# item = ["CO6-010714", "CO6-010334"]
# fill_items(item)

#* function2
embed_size_check()





รายงานจำนวนtabs
counter[item] คือไร:  2
counter[item] คือไร:  2
มี tabs ไรบ้าง {'DevTools': 'D1C7E08DF3991B4AA41307732C6BC89A', 'SMCO :: เปิดการขาย': '5CF3715C425851872B44F6D1920D9317', 'SMCO :: พิมพ์ใบเสร็จซ้ำ': 'DB8050D5D258B6064BEC243F2AD9F5FF', 'SMART QUERY': 'CBD9C1C0BD3B875850DC1B66F5547CC7', 'DevTools1': '68AE60A3FBD931E6501024DB001808DA', 'Seller Centre': '06503CA4A779D3CE535C5E78B9FABBE8', 'SMCO :: เปิดการขาย1': '8E4AAAB9AE0705FB7DBCC2D0E5BEE8AE'}


OSError: [WinError 1155] No application is associated with the specified file for this operation: 'output.pdf'

### PDF READER

In [10]:
from pypdf import PdfReader
import pandas as pd
import re
from openpyxl import load_workbook
import os

extracted_txt:str =""
target_dir = r"TRB018324081500044-Tranfer.pdf"
reader = PdfReader(target_dir)
#* โหลดไฟล์ Excel ที่มีอยู่แล้ว
output_excel = r"Accel_mode.xlsx"

#* สกัดเอา ข้อความออกมาจากไฟล์
for page in reader.pages:
    extracted_txt += page.extract_text()
    
pattern = r'^.*?Product Code Barcode Product Name Transfer No\. Order Ship Status'
extracted_txt = re.sub(pattern, '', extracted_txt, flags=re.DOTALL)
extracted_txt = extracted_txt.lstrip()

pattern2 = r'ผู้ส่งสินค้า.*?(?:No\. Product Code Barcode Product Name Transfer No\. Order Ship Status|วันที่ _ _ _ / _ _ _ / _ _ _)'
extracted_txt = re.sub(pattern2, '', extracted_txt, flags=re.DOTALL)

pattern_serial = r'Serial\s:'
extracted_txt = re.sub(pattern_serial, '', extracted_txt, flags=re.DOTALL)

pattern_sku_no = r'\d+\s{0,}(?=([A-Z0-9]{3}-[0-9]{6}))'
extracted_txt = re.sub(pattern_sku_no, '', extracted_txt, flags=re.DOTALL)

print("อ่านค่าจาก", target_dir)
print(extracted_txt)


#* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
#* Regular expression สำหรับการจับ SKU
sku_pattern = r'([A-Z0-9]{3}-[0-9]{6})'

# *Regular expression สำหรับการจับ serial numbers
serial_pattern = r'Shipped\s*([\w, \n]+)(?=(?:[A-Z0-9]{3}-[0-9]{6}|\nผู้ส่งสินค้า|$))'

#* สกัด SKU
product_codes = re.findall(sku_pattern, extracted_txt)

#* สกัด serial numbers
serial_numbers = re.findall(serial_pattern, extracted_txt, re.DOTALL)

print(serial_numbers)

cleaned_serial_numbers = []
for serial in serial_numbers:
    #* ลบช่องว่างและเลขลำดับที่ไม่ต้องการออก
    cleaned_serial = re.sub(r'\n', '', serial).strip()  #* ลบเลขลำดับที่ท้าย
    # cleaned_serial = re.sub(r'\s+', '', cleaned_serial)  #* ลบช่องว่างทั้งหมด
    cleaned_serial_numbers.append(cleaned_serial)

#* แสดงผล
print("Product Codes:")
code_count = 0
for code in product_codes:
    code_count+=1
    print(code_count, " ", code)

print("\nSerial Numbers:")
code_count = 0
for serial in cleaned_serial_numbers:
    code_count+=1
    #* ลบช่องว่างและเพิ่มวงเล็บ [] รอบ Serial Numbers
    serial = serial.replace(" ", "")
    serial_list = serial.split(",")
    print(f"{code_count} {len(serial_list)} [{serial}]")



#* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
# serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_numbers]
serial_numbers_grouped = [re.findall(r'\b[\w]+\b', serial) for serial in cleaned_serial_numbers]

# ตรวจสอบข้อมูลที่ถูกสกัด
print("SKU Matches:")
print(len(product_codes),product_codes)
print("Serial Numbers Grouped:")
print(len(serial_numbers_grouped), serial_numbers_grouped)

#* สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
data = {sku: serials for sku, serials in zip(product_codes, serial_numbers_grouped)}

# ตรวจสอบ DataFrame ก่อนเขียนลงไฟล์
print("DataFrame:")


#* เอาเข้าตาราง
try:
    # โหลด workbook และ sheet ล่าสุด
    book = load_workbook(output_excel)
    sheet = book.active

    # หาคอลัมน์ล่าสุดที่มีข้อมูล
    last_column = sheet.max_column
    
    # เขียนข้อมูลลงใน Excel
    for col, (sku, serials) in enumerate(data.items(), start=last_column+1):
        sheet.cell(row=1, column=col, value=sku)
        for row, serial in enumerate(serials, start=2):
            sheet.cell(row=row, column=col, value=serial)

    # บันทึกไฟล์
    book.save(output_excel)
    print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")
except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

อ่านค่าจาก TRB018324081500044-Tranfer.pdf
MNL-001906 ACER Gaming Monitor VG270 M3bmiipx -
27"/IPS/180Hz/3Y2 2Shipped
 
 
MMTDPST003423064E04229, MMTDPST003423064C04229
MNL-001939 LENOVO LED Monitor  L24i-40 -
23.8"/IPS/100Hz/3Y*35 5Shipped
 
 
SURP06P2B, SURP06NT9, SURP06P3Y, SURP06P31, SURP06P34
MNL-001941 HP Gaming Monitor OMEN 24 -
23.8"/IPS/165Hz/3Y3 3Shipped
 
 
CNC41535RR, CNC4021CDC, CNC41535R9
MNL-002051 MSI Monitor PRO MP225 -
21.5"/IPS/100Hz/3Y*310 10Shipped
 
 
PC6M064502318, PC6M064502198, PC6M064504367, PC6M064504432, PC6M064504452, PC6M064502283, PC6M064504383,
PC6M064502370, PC6M064502233, PC6M064502201
PR2-000471 BROTHER HL-1110 (2Y*) 1 1Shipped
 
 
E72063M3X928289
PR5-000548 CANON PIXMA G3010 1 1Shipped
 
 
912315C01292AC21KPGR77746
PR5-000610 EPSON Printer L3210 STD/C11CJ68501 70 70Shipped
 
 
XAGH259045, XAGH259053, XAGH259021, XAGH259016, XAGH259056, XAGH259051, XAGH259026, XAGH259050, XAGH259018,
XAGH259022, XAGH257422, XAGH259029, XAGH259049, XAGH259041, XAGH25905

In [33]:
from tkinter import filedialog




def sn_extractor(output_excel, target_dir):
    extracted_txt:str = ""
    # target_dir = r"C:\Users\ONLINE_MIS\Downloads\TRB018324080900002-Tranfer.pdf" //example
    # target_dir = target_dir
    reader = PdfReader(target_dir)
    #* โหลดไฟล์ Excel ที่มีอยู่แล้ว
    # output_excel = r"C:\Users\ONLINE_MIS\Downloads\Accel_mode.xlsx" //example
    # output_excel = output_excel

    #* สกัดเอา ข้อความออกมาจากไฟล์
    for page in reader.pages:
        extracted_txt += page.extract_text()
        
    pattern = r'^.*?(?=No\. Product Code Barcode Product Name Transfer No\. Order Ship Status)'
    extracted_txt = re.sub(pattern, '', extracted_txt, flags=re.DOTALL)
    extracted_txt = extracted_txt.lstrip()
    
    # print(extracted_txt)
    
    #* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
    #* Regular expression สำหรับการจับ SKU
    sku_pattern = r'([A-Z0-9]{3}-[0-9]{6})'

    # *Regular expression สำหรับการจับ serial numbers
    serial_pattern = r'Shipped\s+([\w,\s]+)(?=Serial\s*:)'

    #* สกัด SKU
    sku_matches = re.findall(sku_pattern, extracted_txt)

    #* สกัด serial numbers
    serial_matches = re.findall(serial_pattern, extracted_txt, re.DOTALL)

    #* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
    # serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_matches]
    serial_numbers_grouped = [re.findall(r'\b[\w]+\b', serial) for serial in serial_matches]

    # ตรวจสอบข้อมูลที่ถูกสกัด
    print("SKU Matches:")
    print(len(sku_matches),sku_matches)
    print("Serial Numbers Grouped:")
    print(len(serial_numbers_grouped), serial_numbers_grouped)

    #* สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
    data = {sku: serials for sku, serials in zip(sku_matches, serial_numbers_grouped)}

    # ตรวจสอบ DataFrame ก่อนเขียนลงไฟล์
    print("DataFrame:")


    #* เอาเข้าตาราง
    try:
        # โหลด workbook และ sheet ล่าสุด
        book = load_workbook(output_excel)
        sheet = book.active

        # หาคอลัมน์ล่าสุดที่มีข้อมูล
        last_column = sheet.max_column
        
        # เขียนข้อมูลลงใน Excel
        for col, (sku, serials) in enumerate(data.items(), start=last_column+1):
            sheet.cell(row=1, column=col, value=sku)
            for row, serial in enumerate(serials, start=2):
                sheet.cell(row=row, column=col, value=serial)

        # บันทึกไฟล์
        book.save(output_excel)
        print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")
    except Exception as e:
        print(f"เกิดข้อผิดพลาด: {e}")
        import traceback
        traceback.print_exc()

def extract_sn_btn(accel_file_dir):
    if not accel_file_dir:
        print("select accel file first!!")
        return
    
    target_dirs:tuple = filedialog.askopenfilenames()
    if len(target_dirs) != 0:
        for target_dir in target_dirs:
            sn_extractor(accel_file_dir, target_dir)
    else:
        print("You have not selected any transfer file, Extraction ends!!")

# Test Loguru


In [17]:
from loguru import logger
import threading
orders = ["order1", "order2", "order3", "order4", "order5", "order6", "order7", "order8", "order9", "order10"]



def operation_start(order):
    logger.add("autopageMKII_jupyter_log.log", format="{time} {level} {message}", level="INFO")
    logger.info(f"{order}  Start!!")
    logger.info(f"{order}  Stop!!")
    


for order in orders:
    test_thread = threading.Thread(target=lambda: operation_start(order), name="x")
    
    test_thread.start()
    print(test_thread.is_alive(), "B4 join")
    test_thread.join()
    print(test_thread.is_alive(), "After join")


print(test_thread.name)

2024-08-23 14:59:44.702 | INFO     | __main__:operation_start:9 - order1  Start!!
2024-08-23 14:59:44.719 | INFO     | __main__:operation_start:10 - order1  Stop!!
2024-08-23 14:59:44.722 | INFO     | __main__:operation_start:9 - order2  Start!!
2024-08-23 14:59:44.733 | INFO     | __main__:operation_start:10 - order2  Stop!!
2024-08-23 14:59:44.733 | INFO     | __main__:operation_start:9 - order3  Start!!
2024-08-23 14:59:44.749 | INFO     | __main__:operation_start:10 - order3  Stop!!
2024-08-23 14:59:44.755 | INFO     | __main__:operation_start:9 - order4  Start!!
2024-08-23 14:59:44.760 | INFO     | __main__:operation_start:10 - order4  Stop!!
2024-08-23 14:59:44.765 | INFO     | __main__:operation_start:9 - order5  Start!!
2024-08-23 14:59:44.765 | INFO     | __main__:operation_start:10 - order5  Stop!!
2024-08-23 14:59:44.780 | INFO     | __main__:operation_start:9 - order6  Start!!
2024-08-23 14:59:44.780 | INFO     | __main__:operation_start:10 - order6  Stop!!
2024-08-23 14:59

True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
x
